# Fase 1: Exploratieve Data Analyse (EDA)
## Kadaster - Automatisering Rechtsfeiten Herkenning

Doel: Begrijpen van de dataset en deze gereedmaken voor het model.

### Analysepunten:
1. Distributie check: Class imbalance identificeren
2. Lengte analyse: Tekstlengtes voor BERT modellen (512 token limiet)
3. Kwaliteitscheck: Label consistentie
4. Top 20 labels coverage analyse

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from src.data_loader import jsonl_to_dataframe, get_data_statistics, print_statistics

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Data Laden

In [ ]:
# Load data
data_path = "../ai-challenge-data_ai_challenge_data_anonymized_19744.jsonl"
df = jsonl_to_dataframe(data_path)

print(f"Loaded {len(df)} documents")
df.head()

## 2. Basis Statistieken

In [ ]:
stats = get_data_statistics(df)
print_statistics(stats)

## 3. Label Distributie Analyse
### Checking voor class imbalance

In [ ]:
# Get all labels
all_labels = []
for labels in df['rechtsfeitcodes']:
    all_labels.extend(labels)

label_counts = Counter(all_labels)

# Top 20 labels
top_20_labels = label_counts.most_common(20)
labels, counts = zip(*top_20_labels)

# Plot
plt.figure(figsize=(14, 6))
plt.bar(range(len(labels)), counts, color='steelblue')
plt.xlabel('Rechtsfeit Code')
plt.ylabel('Aantal voorkomens')
plt.title('Top 20 Rechtsfeit Codes - Distributie')
plt.xticks(range(len(labels)), labels, rotation=45)
plt.tight_layout()
plt.show()

# Calculate cumulative coverage
total_labels = len(all_labels)
cumulative_coverage = 0
print("\nTop 20 Labels Coverage Analysis:")
print(f"{'Rank':<6} {'Label':<10} {'Count':<10} {'%':<8} {'Cumulative %':<15}")
print("-" * 60)
for i, (label, count) in enumerate(top_20_labels, 1):
    percentage = (count / total_labels) * 100
    cumulative_coverage += percentage
    print(f"{i:<6} {label:<10} {count:<10} {percentage:<8.2f} {cumulative_coverage:<15.2f}")

## 4. Tekstlengte Analyse
### Belangrijk voor BERT models (512 token limiet)

In [ ]:
# Plot text length distribution
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.hist(df['text_length'], bins=50, color='coral', edgecolor='black')
plt.xlabel('Tekstlengte (karakters)')
plt.ylabel('Aantal documenten')
plt.title('Distributie van Tekstlengtes')
plt.axvline(df['text_length'].mean(), color='red', linestyle='--', label=f'Mean: {df["text_length"].mean():.0f}')
plt.axvline(df['text_length'].median(), color='green', linestyle='--', label=f'Median: {df["text_length"].median():.0f}')
plt.legend()

plt.subplot(1, 2, 2)
plt.hist(df['word_count'], bins=50, color='skyblue', edgecolor='black')
plt.xlabel('Aantal woorden')
plt.ylabel('Aantal documenten')
plt.title('Distributie van Woordenaantal')
plt.axvline(df['word_count'].mean(), color='red', linestyle='--', label=f'Mean: {df["word_count"].mean():.0f}')
plt.axvline(df['word_count'].median(), color='green', linestyle='--', label=f'Median: {df["word_count"].median():.0f}')
# Add 512 token reference line
plt.axvline(512, color='orange', linestyle='--', label='BERT limit: 512 tokens')
plt.legend()

plt.tight_layout()
plt.show()

print(f"\nDocumenten langer dan 512 woorden: {(df['word_count'] > 512).sum()} ({(df['word_count'] > 512).sum() / len(df) * 100:.1f}%)")

## 5. Multi-label Analyse
### Hoeveel documenten hebben meerdere labels?

In [ ]:
# Number of labels per document
label_count_dist = df['num_labels'].value_counts().sort_index()

plt.figure(figsize=(10, 5))
plt.bar(label_count_dist.index, label_count_dist.values, color='mediumseagreen')
plt.xlabel('Aantal labels per document')
plt.ylabel('Aantal documenten')
plt.title('Distributie van Aantal Labels per Document')
plt.xticks(label_count_dist.index)
plt.tight_layout()
plt.show()

print("\nMulti-label statistieken:")
for num_labels, count in label_count_dist.items():
    percentage = (count / len(df)) * 100
    print(f"  {num_labels} label(s): {count:,} documenten ({percentage:.2f}%)")

## 6. Conclusies EDA

Belangrijkste bevindingen:

1. **Class Imbalance**: Er is significante class imbalance. Top labels domineren.
2. **Tekstlengte**: Documenten zijn zeer lang (gemiddeld ~3200 woorden). Vrijwel alle documenten overschrijden de 512 token limiet van BERT.
3. **Multi-label**: Meeste documenten hebben 1 label, maar er zijn ook multi-label cases.
4. **Top 20 Coverage**: De top 20 labels dekken waarschijnlijk >90% van de data.

### Implicaties voor Modellering:
- Voor baseline model: TF-IDF kan volledige tekst gebruiken (geen token limiet)
- Voor BERT: We moeten een strategie implementeren (eerste 512 tokens, sliding window, of samenvatting)
- Class imbalance: We moeten stratified splitting gebruiken
- Multi-label: We kunnen starten met single-label benadering voor simpliciteit

## 7. Sample Document Inspection

In [ ]:
# Look at a sample document
sample = df.iloc[0]
print(f"Akte ID: {sample['akteId']}")
print(f"Rechtsfeit Codes: {sample['rechtsfeitcodes']}")
print(f"Text length: {sample['text_length']} chars")
print(f"Word count: {sample['word_count']} words")
print(f"\nFirst 500 characters of text:\n{sample['text'][:500]}...")